In [1]:
# Cell 0: Environment Hygiene & Setup
# Always uninstall conflicting audio packages before vLLM on Colab T4
!pip uninstall -y torchaudio
!pip install -q vllm

!nvidia-smi


Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 90.2 MB

In [ ]:
# Cell 1: Launch Baseline Server ? Chunked Prefill OFF
# Monolithic prefill: incoming prompts lock the GPU and stall active decodes
!nohup vllm serve Qwen/Qwen3-4B-AWQ --dtype float16 --no-enable-chunked-prefill --port 8000 > vllm_no_chunk.log 2>&1 &
!curl --retry 60 --retry-delay 10 --retry-all-errors -s http://localhost:8000/health


In [4]:
# Cell 2: Victim + Cannon Probe (Chunked Prefill OFF)
# Victim streams tokens continuously while Cannon fires a burst of 2,048-token prompts
import time, threading, requests, json
import numpy as np

VICTIM_PROMPT = "Explain the history and engineering of the steam engine in 300 words."
CANNON_PROMPT = "Explain theoretical physics and general relativity in great detail. " * 200  # ~2,000 tokens

itl_no_chunk = []

def run_victim():
    url = "http://localhost:8000/v1/completions"
    payload = {
        "model": "Qwen/Qwen3-4B-AWQ",
        "prompt": VICTIM_PROMPT,
        "max_tokens": 150,
        "temperature": 0.0,
        "stream": True
    }
    t_prev = None
    with requests.post(url, json=payload, stream=True) as r:
        for line in r.iter_lines():
            t_now = time.perf_counter()
            if line:
                line_str = line.decode('utf-8')
                if line_str.startswith('data: ') and not line_str.endswith('[DONE]'):
                    if t_prev is not None:
                        itl_no_chunk.append((t_now - t_prev) * 1000.0)  # ms
                    t_prev = t_now

def fire_cannon():
    url = "http://localhost:8000/v1/completions"
    payload = {
        "model": "Qwen/Qwen3-4B-AWQ",
        "prompt": CANNON_PROMPT,
        "max_tokens": 1,
        "temperature": 0.0
    }
    threads = [threading.Thread(target=lambda: requests.post(url, json=payload)) for _ in range(4)]
    for t in threads: t.start()
    for t in threads: t.join()

# Start victim streaming client
v_thread = threading.Thread(target=run_victim)
v_thread.start()

# Wait 0.8s for victim stream to establish steady decoding, then FIRE CANNON!
time.sleep(0.8)
fire_cannon()
v_thread.join()

itl_arr_no = np.array(itl_no_chunk)
print(f"=== CHUNKED PREFILL OFF (BASELINE) ===")
print(f"Victim tokens received: {len(itl_arr_no)}")
print(f"Median ITL:             {np.median(itl_arr_no):.2f} ms")
print(f"P99 ITL:                {np.percentile(itl_arr_no, 99):.2f} ms")
print(f"WORST ITL (STALL):      {np.max(itl_arr_no):.2f} ms")


=== CHUNKED PREFILL OFF (BASELINE) ===
Victim tokens received: 149
Median ITL:             14.39 ms
P99 ITL:                34.28 ms
WORST ITL (STALL):      86.65 ms


In [ ]:
# Cell 3: Native vllm bench serve Cross-Check (Chunked Prefill OFF)
# 32 requests with 1024-token prompts at 2 req/s
!vllm bench serve --model Qwen/Qwen3-4B-AWQ \
  --dataset-name random --random-input-len 1024 --random-output-len 64 \
  --num-prompts 32 --request-rate 2 \
  --save-result --result-dir . --result-filename bench_no_chunk.json


In [30]:
!pkill -9 -f "vllm serve"
!sleep 5
!pgrep -af vllm || echo "no vllm processes"
!nvidia-smi

/bin/bash: line 1: kill: (3212) - No such process
                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      20306 F.... vllm
                     root      20448 F...m VLLM::EngineCor
/dev/nvidiactl:      root      20306 F.... vllm
                     root      20448 F...m VLLM::EngineCor
/dev/nvidia-uvm:     root      20306 F.... vllm
                     root      20448 F...m VLLM::EngineCor
Sat Sep 12 21:32:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MI

In [20]:
# Cell 5: Launch Optimized Server -- Chunked Prefill ON (vLLM V1 default)
!nohup vllm serve Qwen/Qwen3-4B-AWQ --dtype float16 \
  --max-model-len 4096 --gpu-memory-utilization 0.9 \
  --max-num-batched-tokens 512 > vllm_on.log 2>&1 &

import time, requests
print('Waiting for vLLM server to start with Chunked Prefill (up to 180s)...')
ready = False
for i in range(36):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'vLLM Server is LIVE and healthy! (took ~{i*5}s)')
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    print('ERROR: Server failed to start! Last 30 lines of vllm_chunked.log:')
    !cat vllm_chunked.log | tail -n 30


Waiting for vLLM server to start with Chunked Prefill (up to 180s)...
vLLM Server is LIVE and healthy! (took ~155s)


In [29]:
# Cell 6: Victim + Cannon Probe (Chunked Prefill ON)
import time, threading, requests, json
import numpy as np

# 1. Verify server is healthy
try:
    r = requests.get('http://localhost:8000/health', timeout=3)
    assert r.status_code == 200, 'Server returned non-200'
    print('Server health confirmed. Starting probe...')
except Exception as e:
    raise RuntimeError(f'Server is NOT reachable on port 8000: {e}. Check Cell 5 log!')

VICTIM_PROMPT = 'Explain the history and engineering of the steam engine in 300 words.'
CANNON_PROMPT = 'Explain theoretical physics and general relativity in great detail. ' * 200

itl_chunked = []

def run_victim_chunked():
    url = 'http://localhost:8000/v1/completions'
    payload = {
        'model': 'Qwen/Qwen3-4B-AWQ',
        'prompt': VICTIM_PROMPT,
        'max_tokens': 150,
        'temperature': 0.0,
        'stream': True
    }
    t_prev = None
    with requests.post(url, json=payload, stream=True) as r:
        for line in r.iter_lines():
            t_now = time.perf_counter()
            if line:
                line_str = line.decode('utf-8')
                if line_str.startswith('data: ') and not line_str.endswith('[DONE]'):
                    if t_prev is not None:
                        itl_chunked.append((t_now - t_prev) * 1000.0)
                    t_prev = t_now

def fire_cannon():
    url = 'http://localhost:8000/v1/completions'
    payload = {
        'model': 'Qwen/Qwen3-4B-AWQ',
        'prompt': CANNON_PROMPT,
        'max_tokens': 1,
        'temperature': 0.0
    }
    threads = [threading.Thread(target=lambda: requests.post(url, json=payload)) for _ in range(4)]
    for t in threads: t.start()
    for t in threads: t.join()

# Start victim streaming client
v_thread2 = threading.Thread(target=run_victim_chunked)
v_thread2.start()

# Wait 0.8s, then FIRE CANNON!
time.sleep(0.8)
fire_cannon()
v_thread2.join()

if len(itl_chunked) == 0:
    print('ERROR: 0 tokens received from victim stream. Check server log!')
else:
    itl_arr_chunk = np.array(itl_chunked)
    print(f'=== CHUNKED PREFILL ON (OPTIMIZED) ===')
    print(f'Victim tokens received: {len(itl_arr_chunk)}')
    print(f'Median ITL:             {np.median(itl_arr_chunk):.2f} ms')
    print(f'P99 ITL:                {np.percentile(itl_arr_chunk, 99):.2f} ms')
    print(f'WORST ITL (STALL):      {np.max(itl_arr_chunk):.2f} ms')


Server health confirmed. Starting probe...
=== CHUNKED PREFILL ON (OPTIMIZED) ===
Victim tokens received: 149
Median ITL:             14.50 ms
P99 ITL:                56.74 ms
WORST ITL (STALL):      82.84 ms


In [32]:
# Cell 7: Native vllm bench serve Cross-Check (Chunked Prefill ON)
# Same 32 requests with 1024-token prompts at 2 req/s
!vllm bench serve --model Qwen/Qwen3-4B-AWQ \
  --dataset-name random --random-input-len 1024 --random-output-len 64 \
  --num-prompts 32 --request-rate 2 \
  --save-result --result-dir . --result-filename bench_chunked.json


Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7cbde0f44e00>, trust_remote_code=False, seed=0, num_prompts=32, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=64, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=